# Alpamayo 1.5: Final Front Camera Projection Notebook

Daily-use final version with navigation-vs-no-navigation comparison.

Features:
1. Uses `Front_camera_original_intrinsic.txt` by default
2. Overlays CoT and metrics on the projected image
3. Shows a clean multi-panel view:
   - front camera image with GT / no-nav Pred / nav Pred projection
   - BEV trajectory plot
   - GT / no-nav Pred / nav Pred speed-accel-yaw comparison plot
4. Supports both single-window and sliding-window testing
5. Sliding mode uses the maximum inference-available t0 range and can run to the latest timestamps
6. Each window runs two conditions:
   - no navigation
   - fixed navigation text
7. In sliding mode, automatically saves each rendered figure to `inference_result/` under the camera-video directory
8. Detailed GT / Pred trajectory tables can be turned on/off by a config switch

Navigation text in this notebook is fixed globally as:
`Turn right at the upcoming intersection`


In [ ]:
import os
import sys
import textwrap
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import av
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
import pandas as pd
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    rotate_xyz_xy_plane,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)

print('repo_root =', repo_root)
print('src_path =', src_path)


In [ ]:
# ===== Reproducibility =====
SEED = 42
set_reproducible_seeds(SEED)

# ===== Paths =====
CLIP_DIR = Path('/workspace/dataset/')
CALIB_DIR = repo_root / 'calibration'
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Calibration =====
FRONT_K_ORIG_PATH = CALIB_DIR / 'Front_camera_original_intrinsic.txt'
FRONT_EXT_PATH = CALIB_DIR / 'Front_camera_extrinsics.txt'
FRONT_DIST_PATH = CALIB_DIR / 'Front_camera_distortion.txt'

# ===== Camera =====
TARGET_CAMERA_FILENAME = 'Front_camera.mp4'

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)
EVAL_XY_ROTATION_DEG = -90.0
FRONT_FRAME_T = -1

# ===== Navigation =====
NAV_TEXT = 'Turn right at the upcoming intersection'

# ===== CoT overlay formatting =====
COT_WRAP_WIDTH = 64
COT_MAX_CHARS = 700

# ===== Run mode =====
RUN_MODE = 'sliding'  # 'single' or 'sliding'
SINGLE_T0_SOD = 175500.23

# ===== Sliding-window settings =====
SLIDING_T0_SOD_LIST = []
SLIDING_STEP_SEC = 1.0
MAX_SLIDING_WINDOWS = None  # e.g. 10 for quick debugging

# ===== Figure saving =====
SAVE_FIGURES_IN_SLIDING_MODE = True
SAVE_FIG_DPI = 140
SAVE_DIR_NAME = 'inference_result'

# ===== BEV style =====
EGO_BOX_LENGTH = 4.5
EGO_BOX_WIDTH = 2.0

# ===== Projection mode =====
FORCE_EXTRINSIC_MODE = 'inverse'  # 'inverse', 'forward', or None for auto

# ===== Table printing =====
PRINT_TRAJECTORY_TABLES = False

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('SEED =', SEED)
print('RUN_MODE =', RUN_MODE)
print('CLIP_DIR =', CLIP_DIR)
print('CALIB_DIR =', CALIB_DIR)
print('TARGET_CAMERA_FILENAME =', TARGET_CAMERA_FILENAME)
print('NAV_TEXT =', NAV_TEXT)
print('FORCE_EXTRINSIC_MODE =', FORCE_EXTRINSIC_MODE)
print('PRINT_TRAJECTORY_TABLES =', PRINT_TRAJECTORY_TABLES)


### Helper functions


In [ ]:
def load_matrix_txt(path: Path):
    arr = np.loadtxt(path, dtype=np.float64)
    print(f'Loaded {path.name}: shape={arr.shape}')
    return arr


def transform_points_homogeneous(points_xyz: np.ndarray, T: np.ndarray) -> np.ndarray:
    pts_h = np.concatenate(
        [points_xyz.astype(np.float64), np.ones((len(points_xyz), 1), dtype=np.float64)],
        axis=1,
    )
    out_h = (T @ pts_h.T).T
    return out_h[:, :3]


def project_points_pinhole(points_cam: np.ndarray, K: np.ndarray):
    X = points_cam[:, 0]
    Y = -points_cam[:, 1]
    Z = points_cam[:, 2]

    valid = Z > 1e-6
    u = np.full_like(X, np.nan, dtype=np.float64)
    v = np.full_like(Y, np.nan, dtype=np.float64)

    u[valid] = K[0, 0] * X[valid] / Z[valid] + K[0, 2]
    v[valid] = K[1, 1] * Y[valid] / Z[valid] + K[1, 2]
    return np.stack([u, v], axis=-1), valid


def decode_single_frame(video_path: Path, frame_index: int) -> np.ndarray:
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for idx, frame in enumerate(container.decode(stream)):
            if idx == frame_index:
                return frame.to_ndarray(format='rgb24')
    raise IndexError(f'Frame index {frame_index} out of range for {video_path}')


def pick_camera_row_by_exact_filename(clip_dir: Path, filename: str):
    camera_paths = helper.discover_offline_camera_files(clip_dir)
    camera_paths_sorted = sorted(camera_paths, key=lambda p: helper.infer_camera_index(p.name))
    matches = [(row, p) for row, p in enumerate(camera_paths_sorted) if p.name == filename]

    if len(matches) == 0:
        raise ValueError(f'No camera file matched exact filename={filename!r}')
    if len(matches) > 1:
        raise ValueError(f'Multiple camera files matched exact filename={filename!r}')

    row, path = matches[0]
    return row, path


def choose_reasonable_extrinsic_direction(points_body: np.ndarray, T_candidate: np.ndarray, K: np.ndarray, image_shape_hw):
    h, w = image_shape_hw

    def score(T):
        pts_cam = transform_points_homogeneous(points_body, T)
        uv, valid_z = project_points_pinhole(pts_cam, K)
        valid_uv = (
            valid_z
            & np.isfinite(uv[:, 0])
            & np.isfinite(uv[:, 1])
            & (uv[:, 0] >= 0)
            & (uv[:, 0] < w)
            & (uv[:, 1] >= 0)
            & (uv[:, 1] < h)
        )
        return int(valid_uv.sum()), uv, pts_cam, valid_uv

    score_fwd, uv_fwd, pts_cam_fwd, valid_fwd = score(T_candidate)
    score_inv, uv_inv, pts_cam_inv, valid_inv = score(np.linalg.inv(T_candidate))

    if score_inv > score_fwd:
        return np.linalg.inv(T_candidate), uv_inv, pts_cam_inv, valid_inv, 'inverse'
    return T_candidate, uv_fwd, pts_cam_fwd, valid_fwd, 'forward'


def resolve_extrinsic_mode(points_body: np.ndarray, T_front: np.ndarray, K_front_orig: np.ndarray, image_shape_hw, force_mode=None):
    if force_mode == 'inverse':
        T_used = np.linalg.inv(T_front)
        pts_cam = transform_points_homogeneous(points_body, T_used)
        uv, valid = project_points_pinhole(pts_cam, K_front_orig)
        return T_used, uv, pts_cam, valid, 'inverse'
    if force_mode == 'forward':
        T_used = T_front
        pts_cam = transform_points_homogeneous(points_body, T_used)
        uv, valid = project_points_pinhole(pts_cam, K_front_orig)
        return T_used, uv, pts_cam, valid, 'forward'
    return choose_reasonable_extrinsic_direction(points_body, T_front, K_front_orig, image_shape_hw)


def plotting_frame_to_t0_local(xyz_plot: np.ndarray, eval_xy_rotation_deg: float) -> np.ndarray:
    return rotate_xyz_xy_plane(xyz_plot, -eval_xy_rotation_deg)


def convert_t0_local_to_camera_body_frame(xyz_t0_local: np.ndarray) -> np.ndarray:
    R = np.array([
        [0.0,  1.0,  0.0],
        [-1.0, 0.0,  0.0],
        [0.0,  0.0, -1.0],
    ], dtype=np.float64)
    return xyz_t0_local @ R.T


def extract_metric_text(df_metrics: pd.DataFrame, key_candidates):
    if df_metrics is None or len(df_metrics) == 0:
        return 'N/A'
    row = df_metrics.iloc[0]
    for key in key_candidates:
        if key in row.index:
            value = row[key]
            if pd.notna(value):
                return f'{float(value):.4f}'
    return 'N/A'


def format_condition_text(label, cot_value, minade_text, final_err_text, wrap_width=64, max_chars=700):
    cot_part = '<empty>' if cot_value is None else str(cot_value).strip()
    if len(cot_part) > max_chars:
        cot_part = cot_part[:max_chars] + ' ...'
    cot_wrapped = textwrap.fill(cot_part, width=wrap_width)
    return (
        f'{label}\n'
        f'minADE: {minade_text}\n'
        f'final_err: {final_err_text}\n\n'
        f'CoT:\n{cot_wrapped}'
    )


def wrap_angle_deg(angle_deg: np.ndarray) -> np.ndarray:
    return (angle_deg + 180.0) % 360.0 - 180.0


def compute_per_step_kinematics(xyz: np.ndarray, dt: float, prefix: str) -> pd.DataFrame:
    xyz = np.asarray(xyz, dtype=np.float64)
    n = len(xyz)

    out = pd.DataFrame({
        't_sec': np.arange(n, dtype=np.float64) * float(dt),
        f'{prefix}_x': xyz[:, 0],
        f'{prefix}_y': xyz[:, 1],
    })

    speed = np.full(n, np.nan, dtype=np.float64)
    accel = np.full(n, np.nan, dtype=np.float64)
    yaw_deg = np.full(n, np.nan, dtype=np.float64)

    if n >= 2:
        dxyz = xyz[1:] - xyz[:-1]
        vel_seg = dxyz / float(dt)
        spd_seg = np.linalg.norm(vel_seg[:, :2], axis=1)
        yaw_seg = np.degrees(np.arctan2(dxyz[:, 1], dxyz[:, 0]))

        speed[:-1] = spd_seg
        speed[-1] = spd_seg[-1]
        yaw_deg[:-1] = yaw_seg
        yaw_deg[-1] = yaw_seg[-1]

        if n >= 3:
            accel_seg = np.diff(spd_seg) / float(dt)
            accel[1:-1] = accel_seg
            accel[0] = accel_seg[0]
            accel[-1] = accel_seg[-1]
        else:
            accel[:] = 0.0

    out[f'{prefix}_speed_post'] = speed
    out[f'{prefix}_accel_post'] = accel
    out[f'{prefix}_yaw_post'] = wrap_angle_deg(yaw_deg)
    return out


def build_condition_trajectory_table(wr, key, dt: float) -> pd.DataFrame:
    gt_df = compute_per_step_kinematics(wr['gt_bev_valid'], dt=dt, prefix='gt')
    pred_df = compute_per_step_kinematics(wr[key]['pred_bev'], dt=dt, prefix=key)
    df = gt_df.merge(pred_df, on='t_sec', how='outer')
    return df


def compute_inference_capable_t0_range(cache, *, num_history_steps, time_step, num_frames, fps, frame0_gps_time_sod):
    pose_min = float(cache.pose_times_sod[0])
    pose_max = float(cache.pose_times_sod[-1])

    history_left_span = (num_history_steps - 1) * time_step
    pose_t0_min = pose_min + history_left_span
    pose_t0_max = pose_max

    image_left_span = (num_frames - 1) * time_step
    image_t0_min = frame0_gps_time_sod + image_left_span
    return pose_t0_min, pose_t0_max, image_t0_min


def build_t0_list(run_mode, single_t0_sod, sliding_t0_sod_list, cache, *, num_history_steps, time_step, num_frames, fps, frame0_gps_time_sod, front_video_path, max_windows, sliding_step_sec):
    if run_mode == 'single':
        return [float(single_t0_sod)]

    if sliding_t0_sod_list:
        t0_list = [float(x) for x in sliding_t0_sod_list]
    else:
        pose_t0_min, pose_t0_max, image_t0_min = compute_inference_capable_t0_range(
            cache,
            num_history_steps=num_history_steps,
            time_step=time_step,
            num_frames=num_frames,
            fps=fps,
            frame0_gps_time_sod=frame0_gps_time_sod,
        )

        with av.open(str(front_video_path)) as container:
            stream = container.streams.video[0]
            num_video_frames = stream.frames
            if not num_video_frames or num_video_frames <= 0:
                num_video_frames = sum(1 for _ in container.decode(stream))

        image_t_last = frame0_gps_time_sod + (num_video_frames - 1) / fps
        image_t0_max = image_t_last

        t0_min = max(pose_t0_min, image_t0_min)
        t0_max = min(pose_t0_max, image_t0_max)

        if t0_max < t0_min:
            raise ValueError(f'No valid sliding t0 range. Computed range: [{t0_min:.3f}, {t0_max:.3f}]')

        t0_list = list(np.arange(t0_min, t0_max + 1e-9, sliding_step_sec, dtype=np.float64))

    if max_windows is not None:
        t0_list = t0_list[:max_windows]
    return t0_list


def draw_projection_panel(ax, image, no_nav_uv, no_nav_valid, nav_uv, nav_valid, gt_uv, gt_valid, title, overlay_text, image_w, image_h):
    ax.imshow(image)

    if gt_valid.any():
        ax.plot(gt_uv[gt_valid, 0], gt_uv[gt_valid, 1], 'k-o', linewidth=2.5, markersize=4, label='GT(valid)')

    if no_nav_valid.any():
        ax.plot(no_nav_uv[no_nav_valid, 0], no_nav_uv[no_nav_valid, 1], 'r-o', linewidth=2.2, markersize=4, label='Pred no-nav')

    if nav_valid.any():
        ax.plot(nav_uv[nav_valid, 0], nav_uv[nav_valid, 1], 'g-o', linewidth=2.2, markersize=4, label='Pred nav')

    ax.set_xlim(0, image_w)
    ax.set_ylim(image_h, 0)
    ax.set_title(title)
    ax.legend(loc='upper right')

    ax.text(
        0.02, 0.98, overlay_text,
        transform=ax.transAxes,
        va='top', ha='left', fontsize=8.5, color='white',
        bbox=dict(boxstyle='round', facecolor='black', alpha=0.65),
    )


def draw_bev_panel(ax, wr):
    hist_xy = wr['hist_bev'][:, :2]
    hist_plot = np.stack([-hist_xy[:, 1], hist_xy[:, 0]], axis=1)

    arrays = [hist_plot]
    if len(wr['gt_bev_valid']) > 0:
        gt_xy = wr['gt_bev_valid'][:, :2]
        gt_plot = np.stack([-gt_xy[:, 1], gt_xy[:, 0]], axis=1)
        arrays.append(gt_plot)
    else:
        gt_plot = None

    no_nav_xy = wr['no_nav']['pred_bev'][:, :2]
    nav_xy = wr['with_nav']['pred_bev'][:, :2]
    no_nav_plot = np.stack([-no_nav_xy[:, 1], no_nav_xy[:, 0]], axis=1)
    nav_plot = np.stack([-nav_xy[:, 1], nav_xy[:, 0]], axis=1)
    arrays.extend([no_nav_plot, nav_plot])

    xlim, ylim = _compute_adaptive_xy_limits(*arrays, min_span=20.0, pad_ratio=0.12, pad_min=2.0)

    ax.plot(hist_plot[:, 0], hist_plot[:, 1], 'o-', color='black', linewidth=2.0, markersize=3.0, label='History')
    if gt_plot is not None:
        ax.plot(gt_plot[:, 0], gt_plot[:, 1], 'o-', color='blue', linewidth=2.5, markersize=3.5, label='GT(valid)')
    ax.plot(no_nav_plot[:, 0], no_nav_plot[:, 1], 'o-', color='red', linewidth=2.3, markersize=3.2, label='Pred no-nav')
    ax.plot(nav_plot[:, 0], nav_plot[:, 1], 'o-', color='green', linewidth=2.3, markersize=3.2, label='Pred nav')

    ego_rect = patches.Rectangle(
        (-EGO_BOX_WIDTH / 2.0, -EGO_BOX_LENGTH / 2.0),
        EGO_BOX_WIDTH,
        EGO_BOX_LENGTH,
        linewidth=2.0,
        edgecolor='red',
        facecolor='none',
        label='Ego',
        zorder=4,
    )
    ax.add_patch(ego_rect)

    ax.set_xlabel('lateral (sign-flipped for visualization)')
    ax.set_ylabel('forward')
    ax.set_title('BEV trajectory plot')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='best')


def draw_kinematics_panel(ax_speed, ax_accel, ax_yaw, wr):
    if len(wr['gt_bev_valid']) > 0:
        kin_gt = compute_per_step_kinematics(wr['gt_bev_valid'], dt=TIME_STEP, prefix='gt')
        t_gt = kin_gt['t_sec'].to_numpy()
        ax_speed.plot(t_gt, kin_gt['gt_speed_post'], color='blue', linewidth=2.0, label='GT speed')
        ax_accel.plot(t_gt, kin_gt['gt_accel_post'], color='blue', linewidth=2.0, label='GT accel')
        ax_yaw.plot(t_gt, kin_gt['gt_yaw_post'], color='blue', linewidth=2.0, label='GT yaw')

    kin_no_nav = compute_per_step_kinematics(wr['no_nav']['pred_bev'], dt=TIME_STEP, prefix='no_nav')
    kin_nav = compute_per_step_kinematics(wr['with_nav']['pred_bev'], dt=TIME_STEP, prefix='nav')

    t_no_nav = kin_no_nav['t_sec'].to_numpy()
    t_nav = kin_nav['t_sec'].to_numpy()

    ax_speed.plot(t_no_nav, kin_no_nav['no_nav_speed_post'], color='red', linewidth=2.0, label='Pred no-nav speed')
    ax_speed.plot(t_nav, kin_nav['nav_speed_post'], color='green', linewidth=2.0, label='Pred nav speed')
    ax_speed.set_ylabel('speed (m/s)')
    ax_speed.grid(True, alpha=0.3)
    ax_speed.legend(loc='best')

    ax_accel.plot(t_no_nav, kin_no_nav['no_nav_accel_post'], color='red', linewidth=2.0, label='Pred no-nav accel')
    ax_accel.plot(t_nav, kin_nav['nav_accel_post'], color='green', linewidth=2.0, label='Pred nav accel')
    ax_accel.set_ylabel('accel (m/s^2)')
    ax_accel.grid(True, alpha=0.3)
    ax_accel.legend(loc='best')

    ax_yaw.plot(t_no_nav, kin_no_nav['no_nav_yaw_post'], color='red', linewidth=2.0, label='Pred no-nav yaw')
    ax_yaw.plot(t_nav, kin_nav['nav_yaw_post'], color='green', linewidth=2.0, label='Pred nav yaw')
    ax_yaw.set_ylabel('yaw (deg)')
    ax_yaw.set_xlabel('t (sec)')
    ax_yaw.grid(True, alpha=0.3)
    ax_yaw.legend(loc='best')


def render_window_result(wr):
    fig = plt.figure(figsize=(22, 12))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.0], width_ratios=[1.15, 1.0])

    ax_proj = fig.add_subplot(gs[0, 0])
    ax_bev = fig.add_subplot(gs[0, 1])
    subgs = gs[1, :].subgridspec(3, 1, hspace=0.18)
    ax_speed = fig.add_subplot(subgs[0, 0])
    ax_accel = fig.add_subplot(subgs[1, 0])
    ax_yaw = fig.add_subplot(subgs[2, 0])

    draw_projection_panel(
        ax=ax_proj,
        image=wr['front_img_orig'],
        no_nav_uv=wr['no_nav']['uv_pred'],
        no_nav_valid=wr['no_nav']['valid_pred'],
        nav_uv=wr['with_nav']['uv_pred'],
        nav_valid=wr['with_nav']['valid_pred'],
        gt_uv=wr['uv_gt'],
        gt_valid=wr['valid_gt'],
        title=f"Front projection | t0={wr['t0_sod']:.3f} | frame={wr['requested_front_frame_idx']} | extrinsic={wr['extrinsic_mode']}",
        overlay_text=wr['overlay_text'],
        image_w=wr['image_w'],
        image_h=wr['image_h'],
    )

    draw_bev_panel(ax_bev, wr)
    draw_kinematics_panel(ax_speed, ax_accel, ax_yaw, wr)

    plt.tight_layout()
    return fig


def save_window_figure(fig, wr, front_video_path, dpi=140, save_dir_name='inference_result'):
    save_dir = front_video_path.parent / save_dir_name
    save_dir.mkdir(parents=True, exist_ok=True)
    save_name = f"{front_video_path.stem}_projection_t0_{wr['t0_sod']:.3f}.png"
    save_path = save_dir / save_name
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    return save_path


def run_projection_condition(result, *, K_front_orig, T_front, front_img_orig):
    gt_xyz_plot = result['gt_xyz_plot']
    pred_xyz_plot = result['pred_best_xyz']

    gt_xyz_t0_local = plotting_frame_to_t0_local(gt_xyz_plot, EVAL_XY_ROTATION_DEG) if len(gt_xyz_plot) > 0 else np.empty((0, 3), dtype=np.float64)
    pred_xyz_t0_local = plotting_frame_to_t0_local(pred_xyz_plot, EVAL_XY_ROTATION_DEG)

    gt_xyz_body = convert_t0_local_to_camera_body_frame(gt_xyz_t0_local) if len(gt_xyz_t0_local) > 0 else np.empty((0, 3), dtype=np.float64)
    pred_xyz_body = convert_t0_local_to_camera_body_frame(pred_xyz_t0_local)

    image_h, image_w = front_img_orig.shape[:2]
    T_front_used, _, _, _, extrinsic_mode = resolve_extrinsic_mode(
        pred_xyz_body,
        T_front,
        K_front_orig,
        (image_h, image_w),
        force_mode=FORCE_EXTRINSIC_MODE,
    )

    if len(gt_xyz_body) > 0:
        gt_xyz_cam = transform_points_homogeneous(gt_xyz_body, T_front_used)
        uv_gt, valid_gt = project_points_pinhole(gt_xyz_cam, K_front_orig)
    else:
        uv_gt = np.empty((0, 2), dtype=np.float64)
        valid_gt = np.zeros((0,), dtype=bool)

    pred_xyz_cam = transform_points_homogeneous(pred_xyz_body, T_front_used)
    uv_pred, valid_pred = project_points_pinhole(pred_xyz_cam, K_front_orig)

    return {
        'gt_bev_valid': gt_xyz_plot,
        'pred_bev': pred_xyz_plot,
        'uv_gt': uv_gt,
        'valid_gt': valid_gt,
        'uv_pred': uv_pred,
        'valid_pred': valid_pred,
        'extrinsic_mode': extrinsic_mode,
        'num_valid_future_steps': result['num_valid_future_steps'],
        'metrics_df': result['df_metrics'],
        'cot_value': result.get('cot_value', None),
        'metrics_row': result['df_metrics'].iloc[0].to_dict() if len(result['df_metrics']) else {},
        'num_gt_projected_points_in_image': int(valid_gt.sum()),
        'num_pred_projected_points_in_image': int(valid_pred.sum()),
    }


def run_projection_window(t0_sod, *, model, processor, K_front_orig, T_front, clip_dir, front_cam_row, front_video_path):
    data = load_offline_dataset(
        clip_dir=clip_dir,
        t0_sod=float(t0_sod),
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        debug=False,
        image_size=IMAGE_SIZE,
    )

    no_nav_result = run_offline_inference_window(
        data=data,
        model=model,
        processor=processor,
        device=DEVICE,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        time_step=TIME_STEP,
        eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
        nav_text=None,
    )

    nav_result = run_offline_inference_window(
        data=data,
        model=model,
        processor=processor,
        device=DEVICE,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        time_step=TIME_STEP,
        eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
        nav_text=NAV_TEXT,
    )

    requested_front_frame_idx = int(data['actual_video_frame_indices'][front_cam_row, FRONT_FRAME_T].item())
    requested_front_ts = float(data['absolute_timestamps_sod'][front_cam_row, FRONT_FRAME_T].item())
    front_img_orig = decode_single_frame(front_video_path, requested_front_frame_idx)
    image_h, image_w = front_img_orig.shape[:2]

    no_nav_proj = run_projection_condition(no_nav_result, K_front_orig=K_front_orig, T_front=T_front, front_img_orig=front_img_orig)
    nav_proj = run_projection_condition(nav_result, K_front_orig=K_front_orig, T_front=T_front, front_img_orig=front_img_orig)

    gt_bev_valid = no_nav_result['gt_xyz_plot'] if len(no_nav_result['gt_xyz_plot']) > 0 else nav_result['gt_xyz_plot']
    uv_gt = no_nav_proj['uv_gt'] if len(no_nav_proj['uv_gt']) > 0 else nav_proj['uv_gt']
    valid_gt = no_nav_proj['valid_gt'] if len(no_nav_proj['valid_gt']) > 0 else nav_proj['valid_gt']

    no_nav_minade = extract_metric_text(no_nav_proj['metrics_df'], ['minADE_m', 'minADE'])
    no_nav_final = extract_metric_text(no_nav_proj['metrics_df'], ['final_point_error_m'])
    nav_minade = extract_metric_text(nav_proj['metrics_df'], ['minADE_m', 'minADE'])
    nav_final = extract_metric_text(nav_proj['metrics_df'], ['final_point_error_m'])

    overlay_text = (
        format_condition_text('No navigation', no_nav_proj['cot_value'], no_nav_minade, no_nav_final, wrap_width=COT_WRAP_WIDTH, max_chars=COT_MAX_CHARS)
        + '\n\n'
        + format_condition_text(f'With navigation: {NAV_TEXT}', nav_proj['cot_value'], nav_minade, nav_final, wrap_width=COT_WRAP_WIDTH, max_chars=COT_MAX_CHARS)
    )

    return {
        't0_sod': float(t0_sod),
        'data': data,
        'front_img_orig': front_img_orig,
        'image_h': image_h,
        'image_w': image_w,
        'requested_front_frame_idx': requested_front_frame_idx,
        'requested_front_ts': requested_front_ts,
        'extrinsic_mode': no_nav_proj['extrinsic_mode'],
        'overlay_text': overlay_text,
        'valid_gt': valid_gt,
        'uv_gt': uv_gt,
        'gt_bev_valid': gt_bev_valid,
        'hist_bev': no_nav_result['hist_xyz_plot'],
        'no_nav': no_nav_proj,
        'with_nav': nav_proj,
    }


### Load calibration, cache, model, and processor


In [ ]:
for p in [FRONT_K_ORIG_PATH, FRONT_EXT_PATH, FRONT_DIST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing calibration file: {p}')

K_front_orig = load_matrix_txt(FRONT_K_ORIG_PATH)
T_front = load_matrix_txt(FRONT_EXT_PATH)
dist_front = load_matrix_txt(FRONT_DIST_PATH)

clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=False, force_rebuild=False)
front_cam_row, front_video_path = pick_camera_row_by_exact_filename(CLIP_DIR, TARGET_CAMERA_FILENAME)

save_dir = front_video_path.parent / SAVE_DIR_NAME
save_dir.mkdir(parents=True, exist_ok=True)
print('save_dir =', save_dir)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Loaded calibration, cache, model, and processor.')
print('front_cam_row =', front_cam_row)
print('front_video_path =', front_video_path)


### Run single window or sliding-window projection


In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

t0_list = build_t0_list(
    run_mode=RUN_MODE,
    single_t0_sod=SINGLE_T0_SOD,
    sliding_t0_sod_list=SLIDING_T0_SOD_LIST,
    cache=clip_cache,
    num_history_steps=NUM_HISTORY_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    front_video_path=front_video_path,
    max_windows=MAX_SLIDING_WINDOWS,
    sliding_step_sec=SLIDING_STEP_SEC,
)

print('num windows =', len(t0_list))
print('first few t0_sod =', t0_list[:5])
if len(t0_list):
    print('last few t0_sod =', t0_list[-5:])

window_results = []
saved_figure_paths = []

for i, t0_sod in enumerate(t0_list, start=1):
    print(f'\n=== Running window {i}/{len(t0_list)} | t0_sod={t0_sod:.3f} ===')
    wr = run_projection_window(
        t0_sod=t0_sod,
        model=model,
        processor=processor,
        K_front_orig=K_front_orig,
        T_front=T_front,
        clip_dir=CLIP_DIR,
        front_cam_row=front_cam_row,
        front_video_path=front_video_path,
    )
    window_results.append(wr)

    print(
        f"frame_idx={wr['requested_front_frame_idx']}, frame_ts={wr['requested_front_ts']:.3f}, "
        f"extrinsic={wr['extrinsic_mode']}, gt_projected={int(wr['valid_gt'].sum())}, "
        f"no_nav_pred_projected={wr['no_nav']['num_pred_projected_points_in_image']}, "
        f"nav_pred_projected={wr['with_nav']['num_pred_projected_points_in_image']}"
    )

    fig = render_window_result(wr)
    plt.show()

    if RUN_MODE == 'sliding' and SAVE_FIGURES_IN_SLIDING_MODE:
        save_path = save_window_figure(fig, wr, front_video_path, dpi=SAVE_FIG_DPI, save_dir_name=SAVE_DIR_NAME)
        saved_figure_paths.append(str(save_path))
        print('saved figure ->', save_path)

    plt.close(fig)

    if PRINT_TRAJECTORY_TABLES:
        print(f"[Window t0_sod={wr['t0_sod']:.6f}] No-nav GT / Pred trajectory table")
        display(build_condition_trajectory_table(wr, 'no_nav', dt=TIME_STEP))
        print(f"[Window t0_sod={wr['t0_sod']:.6f}] Nav GT / Pred trajectory table")
        display(build_condition_trajectory_table(wr, 'with_nav', dt=TIME_STEP))

print('\nFinished all windows.')


### Concise result summary table


In [ ]:
summary_rows = []
for wr in window_results:
    row = {
        't0_sod': wr['t0_sod'],
        'frame_idx': wr['requested_front_frame_idx'],
        'frame_ts': wr['requested_front_ts'],
        'extrinsic_mode': wr['extrinsic_mode'],
        'num_gt_projected_points_in_image': int(wr['valid_gt'].sum()),
        'no_nav_pred_projected_points': wr['no_nav']['num_pred_projected_points_in_image'],
        'nav_pred_projected_points': wr['with_nav']['num_pred_projected_points_in_image'],
        'no_nav_num_valid_future_steps': wr['no_nav']['num_valid_future_steps'],
        'nav_num_valid_future_steps': wr['with_nav']['num_valid_future_steps'],
    }

    no_nav_metrics = wr['no_nav']['metrics_row']
    nav_metrics = wr['with_nav']['metrics_row']

    row.update({f'no_nav_{k}': v for k, v in no_nav_metrics.items()})
    row.update({f'nav_{k}': v for k, v in nav_metrics.items()})

    row['delta_pred_final_x_nav_minus_no_nav'] = nav_metrics.get('pred_final_x', np.nan) - no_nav_metrics.get('pred_final_x', np.nan)
    row['delta_pred_final_y_nav_minus_no_nav'] = nav_metrics.get('pred_final_y', np.nan) - no_nav_metrics.get('pred_final_y', np.nan)
    row['delta_pred_mean_speed_nav_minus_no_nav'] = nav_metrics.get('pred_mean_speed_mps', np.nan) - no_nav_metrics.get('pred_mean_speed_mps', np.nan)
    row['delta_pred_yaw_nav_minus_no_nav'] = nav_metrics.get('pred_yaw_deg', np.nan) - no_nav_metrics.get('pred_yaw_deg', np.nan)

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

if saved_figure_paths:
    print('\nSaved figure files:')
    for p in saved_figure_paths:
        print(p)
